# 🕸️ Workshop: Advanced GraphRAG with Google Spanner & Gemini
**Duration:** 3 Hours | **Level:** Intermediate to Advanced

**Author:** Hasan Rafiq - Google AI

### 🎯 Objective
Build a complete **GraphRAG** system — entirely inside Google Cloud — using **Cloud Spanner Graph** as the knowledge store and **Gemini** as the reasoning engine. By the end of this workshop you will know how to:
- Extract entities and relationships from unstructured text using Gemini
- Store a Knowledge Graph natively in Cloud Spanner Graph
- Run multi-hop reasoning queries using GQL (Graph Query Language)
- Build a hybrid search pipeline combining vector similarity with graph traversal

### 🧩 Why GraphRAG?
Standard RAG (Vector Search) is great for finding specific facts but fails at:
1. **Multi-hop Queries:** "How does the CEO of Company A relate to the project delay in Department B?"
2. **Global Summarization:** "What are the three main themes across these 500 documents?"
3. **Relationship Discovery:** Identifying hidden connections that aren't mentioned in the same text chunk.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanrafiq-goog/hs-gdb-ws/blob/main/graph_nb_workshop.ipynb)

# Section 1: 🏗️ The GraphRAG Pipeline We Are Building

```
Raw Text (Corporate Memos)
        │
        ▼
  ┌─────────────┐
  │  EXTRACTION │  ← Gemini reads text and identifies Nodes (Entities)
  │             │    and Edges (Relationships)
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │  INDEXING   │  ← Data stored as a Property Graph in Cloud Spanner
  │             │    Vector embeddings stored directly on graph nodes
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │  RETRIEVAL  │  ← Stage 1: Vector search + graph expansion finds subgraph
  │  (Hybrid)   │    Stage 2: Gemini generates a GQL graph traversal query
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │ GENERATION  │  ← Gemini synthesizes a final answer from the sub-graph
  └─────────────┘
```

---

## 🚀 Pre-step: Setting up your GCP Project & Billing

If you don't have a Google Cloud Project with billing enabled, you can use the following referral link to claim credits and set one up:

**[Claim your GCP Credits here](https://trygcp.dev/claim/hsbc-krakow-event)**

### Steps to create a new GCP Project and enable billing:

1.  **Click the referral link above**. This will take you to a form where you get the option to "Click here to access your credits".
2.  Click "Claim your credits". This will take you to a "GCP credit application" page. Enter your details and click "Accept and continue".
3.  You will be redirected to a new GCP billing account where your credits are already available.
4.  **Create a New GCP Project within this billing account:**
    *   Go to the [Google Cloud Console](https://console.cloud.google.com/).
    *   At the top left, click on the project selector dropdown (it usually shows your current project name or "My First Project").
    *   In the project selection dialog, click **"New Project"**.
    *   Give your project a meaningful name (e.g., `graphrag-workshop`).
    *   Ensure the billing account with the claimed credits is selected for this new project.
    *   Click **"Create"**.
5.  Note down your **Project ID** (it's often a combination of your project name and some random characters, like `your-project-name-12345`). This ID will be needed in the configuration cell below.
6.  Once your project is created and billing is enabled, copy your **GCP Project ID** and paste it into the `GCP_PROJECT_ID` field in the **`Configuration`** cell (Step 1.2) below.

## 🗄️ Step 0: Before You Begin

### Prerequisites
1. A **Google Cloud Project** with billing enabled
2. The **Cloud Spanner API** enabled ([enable it here](https://console.cloud.google.com/flows/enableapi?apiid=spanner.googleapis.com))
3. The **Vertex AI API** enabled ([enable it here](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com))
4. Owner or Editor role on the project (or at minimum: `roles/spanner.admin` + `roles/aiplatform.user`)

---

## 📦 Step 1.1: Install Dependencies

In [ ]:
# Install all required libraries
# langchain-google-spanner: LangChain integration for Spanner Graph
# langchain-google-vertexai: Gemini LLM + Embedding models via Vertex AI
# langchain-experimental: LLMGraphTransformer for entity/relationship extraction
# google-cloud-spanner: Native Spanner Python SDK

%pip install --quiet langchain-google-spanner==0.8.2
%pip install --quiet langchain-google-vertexai==2.0.22
%pip install --quiet langchain-experimental==0.3.4
%pip install --quiet langchain-community==0.3.24
%pip install --quiet langchain-core==0.3.59
%pip install --quiet langchain-text-splitters==0.3.8
%pip install --quiet google-cloud-spanner==3.56.0
%pip install --quiet google-cloud-aiplatform==1.121.0
%pip install --quiet pyvis  # For knowledge graph visualization

print("✅ All dependencies installed!")

## 🔐 Step 1.2: Authentication & Configuration

### Why this matters
Because we are using Spanner (a native GCP service), authentication uses your existing GCP credentials and IAM roles. There are **no external API keys, no third-party tokens, no data leaving your project boundary**. Your security team can audit access through standard GCP IAM logs.

In [ ]:
import sys

if 'google.colab' in sys.modules:
    from google.colab import auth as google_auth
    google_auth.authenticate_user()
    print("✅ Authenticated via Colab OAuth")
else:
    print("ℹ️  Running in Vertex AI Workbench — using attached service account credentials")

print(f"\nPython version: {sys.version}")

In [ ]:
import os

# ─────────────────────────────────────────────────────────────
# 🔧 CONFIGURATION — Update these values for your environment
# ─────────────────────────────────────────────────────────────

GCP_PROJECT_ID      = ""  # @param {type:"string"}
REGION              = "us-central1"             # @param {type:"string"}
MODEL_NAME          = "gemini-2.5-flash"        # @param {type:"string"}
EMBEDDING_MODEL     = "text-embedding-004"      # @param {type:"string"}

SPANNER_INSTANCE_ID = "graphrag-workshop"       # @param {type:"string"}
SPANNER_DATABASE_ID = "graphrag"                # @param {type:"string"}
SPANNER_GRAPH_NAME  = "corpgraph"               # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = GCP_PROJECT_ID
os.environ["GOOGLE_CLOUD_REGION"]  = REGION

print(f"✅ Configuration set:")
print(f"   Project:          {GCP_PROJECT_ID}")
print(f"   Region:           {REGION}")
print(f"   LLM Model:        {MODEL_NAME}")
print(f"   Embedding Model:  {EMBEDDING_MODEL}")
print(f"   Spanner Instance: {SPANNER_INSTANCE_ID}")
print(f"   Spanner Database: {SPANNER_DATABASE_ID}")
print(f"   Graph Name:       {SPANNER_GRAPH_NAME}")

## 🗄️ Step 1.3: Provision the Spanner Graph Database

### What is Cloud Spanner Graph?
Cloud Spanner Graph is a **native graph capability built into Cloud Spanner** — not a separate product. It uses standard Spanner tables under the hood but exposes them through a **Property Graph Model** and a graph query language called **GQL (Graph Query Language)**.

### The Property Graph Model

| Concept | Description | Example |
|---|---|---|
| **Node** | An entity — a "noun" | `Person: Sarah Jenkins` |
| **Edge** | A directional relationship between two nodes | `WORKS_AT` |
| **Property** | Metadata attached to a node or edge | `role: "Senior Architect"` |

### 3.1 — Create the Spanner Instance
An **Instance** is a compute allocation. For a workshop, 1 node is sufficient.

In [ ]:
!gcloud config set project {GCP_PROJECT_ID} --quiet
!gcloud services enable spanner.googleapis.com aiplatform.googleapis.com --quiet

!gcloud spanner instances create {SPANNER_INSTANCE_ID} \
    --config=regional-{REGION} \
    --description="GraphRAG Workshop" \
    --nodes=1 \
    --edition=ENTERPRISE \
    --quiet

print(f"\n✅ Spanner instance '{SPANNER_INSTANCE_ID}' created in {REGION}")
print(f"   View at: https://console.cloud.google.com/spanner/instances")

## Step 1.4 — Create the Database

We just need an empty Spanner database. That's it.

`SpannerGraphStore` handles everything else automatically when we call `add_graph_documents` in the next step:
- Creates a **node table** for each entity type (Person, Organization, Location...)
- Creates an **edge table** for each relationship type (WORKS_AT, ADVISES...)
- Creates the **PROPERTY GRAPH DDL** that maps all those tables into a graph

No manual schema definition. No `CREATE TABLE` statements. No separate embedding table. Everything lives natively on the graph nodes.

In [ ]:
!gcloud spanner databases create {SPANNER_DATABASE_ID} \
    --instance={SPANNER_INSTANCE_ID} --quiet

print("✅ Database created")

---
# 🧩 Section 2: The Extraction Engine

## Understanding the Graph Model Before We Extract

### What is a Node? (The "Nouns")
Nodes represent entities. They are the objects in your data.
- **Examples:** A person (*Sarah Jenkins*), a company (*Acme Corp*), a city (*London*)
- **Labels:** Every node has a label that defines its type: `:Person`, `:Organization`, `:Location`

### What is a Relationship? (The "Verbs")
Relationships connect two nodes. They always have a direction.
- `Sarah Jenkins` → **WORKS_AT** → `Acme Corp`
- `Mark Chen` → **ADVISES** → `Sarah Jenkins`

### What is an Ontology?
An **ontology** is the set of rules that constrains what types of nodes and relationships are allowed. Without it, Gemini might invent arbitrary labels and the graph becomes inconsistent.

We define:
- **`allowed_nodes`** — the entity types Gemini is allowed to extract
- **`allowed_relationships`** — the relationship types Gemini is allowed to create

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from langchain_google_vertexai import ChatVertexAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

# Initialize Gemini as our extraction engine
# temperature=0 keeps the extraction factual and deterministic
extractor_llm = ChatVertexAI(
    model_name=MODEL_NAME,
    project=GCP_PROJECT_ID,
    location=REGION,
    temperature=0
)

# Define the ontology — the rules of our knowledge graph
allowed_nodes         = ["Person", "Organization", "Project", "Location", "Technology"]
allowed_relationships = ["WORKS_AT", "PARTNERED_WITH", "LOCATED_IN", "MANAGED_BY", "USES", "ADVISES"]

transformer = LLMGraphTransformer(
    llm=extractor_llm,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships
)

print("✅ Extraction engine ready!")
print(f"   Allowed node types:         {allowed_nodes}")
print(f"   Allowed relationship types: {allowed_relationships}")

## Step 2.1: Running the Extraction

We feed Gemini three unstructured corporate memos. The information needed to answer multi-hop questions is **spread across all three memos** — no single memo contains the complete answer.

**The Challenge:**
If you asked a standard vector RAG system *"Which city is the firm advising the architect at Acme Corp based in?"*, it would need to:
1. Find the memo about the architect (Memo 2 → Sarah Jenkins)
2. Find the memo about who advises her (Memo 3 → Mark Chen)
3. Find the memo about where Mark Chen's firm is (Memo 1 → London)

A vector search returning the top-k most similar chunks will often miss one of these hops.

**What Gemini Does:**
When you run the cell below, `transformer.convert_to_graph_documents` makes Gemini:
1. **Read** all memos holistically
2. **Identify** entities and their types
3. **Infer** relationships from context (e.g., "consultant at" → `WORKS_AT`)
4. **Output** triples in the format `(Subject) --[Relationship]→ (Object)`

In [ ]:
from langchain_core.documents import Document

# Our raw, unstructured dataset — spread across 3 separate memos
# Notice: no single memo contains all the information needed for a multi-hop query
corporate_memos = """
Memo 1: Acme Corp is a technology giant headquartered in San Francisco.
Global Dynamics is a consultancy based in London.

Memo 2: Sarah Jenkins is the Senior Architect at Acme Corp.
She is currently the Lead for 'Project Phoenix'.

Memo 3: Global Dynamics signed a strategic partnership with Acme Corp for AI safety.
Mark Chen is a consultant at Global Dynamics and has been assigned to advise Sarah Jenkins.
"""

documents = [Document(page_content=corporate_memos)]

print("⏳ Gemini is reasoning through the text...")
graph_documents = transformer.convert_to_graph_documents(documents)

print("\n─── 🕸️ Extracted Knowledge Graph Triples ───")
for rel in graph_documents[0].relationships:
    print(f"  ({rel.source.id}) --[{rel.type}]→ ({rel.target.id})")

print("\n─── 📍 Extracted Nodes ───")
for node in graph_documents[0].nodes:
    print(f"  [{node.type}] {node.id}")

## Step 2.2: Storing the Graph in Cloud Spanner

The triples extracted above exist only in notebook memory. We now persist them into **Cloud Spanner Graph** using the `SpannerGraphStore` from `langchain-google-spanner`.

### What SpannerGraphStore Does Under the Hood
The `add_graph_documents` call:
1. Creates **node tables** in Spanner (one per node type: `corpgraph_Person`, `corpgraph_Organization`, etc.)
2. Creates **edge tables** for each relationship type (e.g., `corpgraph_Person_WORKS_AT_Organization`)
3. Creates a **PROPERTY GRAPH** DDL view that maps these tables into the graph model
4. Uses `INSERT OR UPDATE` to avoid creating duplicate nodes

The graph is stored as **standard Spanner tables** — meaning your existing DBA tooling, backups, IAM policies, and monitoring all work exactly as expected.

In [ ]:
from langchain_google_spanner import SpannerGraphStore
from google.cloud import spanner

# Connect to Spanner Graph
graph_store = SpannerGraphStore(
    instance_id=SPANNER_INSTANCE_ID,
    database_id=SPANNER_DATABASE_ID,
    graph_name=SPANNER_GRAPH_NAME,
)

# Connect the native Spanner client for direct queries later
spanner_client = spanner.Client(GCP_PROJECT_ID)
instance        = spanner_client.instance(SPANNER_INSTANCE_ID)
database        = instance.database(SPANNER_DATABASE_ID)

print("⏳ Writing graph to Cloud Spanner...")
for graph_document in graph_documents:
    graph_store.add_graph_documents([graph_document])

print(f"\n✅ Knowledge Graph persisted to Cloud Spanner!")
print(f"   Instance: {SPANNER_INSTANCE_ID}")
print(f"   Database: {SPANNER_DATABASE_ID}")
print(f"   Graph:    {SPANNER_GRAPH_NAME}")

## Step 2.3: Visualizing the Knowledge Graph

Before we start querying, let's inspect what we just stored. We read directly from the relational tables that `SpannerGraphStore` created and render an interactive visualization.

**What to look for:**
- **Central hubs:** Which nodes have the most connections? (Acme Corp, Global Dynamics)
- **Bridge entities:** Mark Chen connects Global Dynamics to Sarah Jenkins — he is the only path between the two corporate entities
- **Colors:** Each node type gets its own color

### Spanner Studio
You can also visualize the graph in the Spanner Console. Navigate to your database and run:
```gql
GRAPH corpgraph
MATCH p = (a)-[e]->(b)
RETURN TO_JSON(p) AS path_json
LIMIT 50
```

In [ ]:
from pyvis.network import Network
from IPython.display import HTML

# ── Step 1: Identify edge tables ─────────────────────────────────
# SpannerGraphStore names tables as: {graph}_{SourceType}_{REL}_{TargetType}
# Node tables are: {graph}_{NodeType}  e.g. corpgraph_Person
# We distinguish them by checking against our known node type names

all_tables_sql = """
SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = ''
ORDER BY TABLE_NAME
"""

node_types_set = {f"{SPANNER_GRAPH_NAME}_{n}" for n in allowed_nodes}

edge_table_names = []
with database.snapshot() as snapshot:
    for row in snapshot.execute_sql(all_tables_sql):
        if row[0] not in node_types_set:
            edge_table_names.append(row[0])

# ── Step 2: Read edges from each edge table ───────────────────────
results = []
for table_name in edge_table_names:
    # Table name pattern: corpgraph_Person_WORKS_AT_Organization
    # Strip graph prefix → Person_WORKS_AT_Organization
    # Split on _ → [Person, WORKS, AT, Organization]
    # Relationship label is everything between first and last part
    stripped = table_name.replace(f"{SPANNER_GRAPH_NAME}_", "", 1)
    parts    = stripped.split("_")
    rel_label = "_".join(parts[1:-1]) if len(parts) > 2 else stripped

    with database.snapshot() as snapshot:
        for row in snapshot.execute_sql(f"SELECT id, target_id FROM {table_name}"):
            results.append({"source": row[0], "label": rel_label, "target": row[1]})

print(f"📊 Retrieved {len(results)} edges")
for r in results:
    print(f"  ({r['source']}) --[{r['label']}]→ ({r['target']})")

# ── Step 3: Build node → type lookup ─────────────────────────────
color_map = {
    "Person":       "#ff7675",
    "Organization": "#74b9ff",
    "Project":      "#55efc4",
    "Location":     "#ffeaa7",
    "Technology":   "#a29bfe",
}

node_type_lookup = {}
for node_type in allowed_nodes:
    table_name = f"{SPANNER_GRAPH_NAME}_{node_type}"
    check_sql  = f"SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '' AND TABLE_NAME = '{table_name}'"
    with database.snapshot() as snapshot:
        exists = list(snapshot.execute_sql(check_sql))[0][0] > 0
    if not exists:
        print(f"  ⚠️  No entities of type '{node_type}' extracted — skipping")
        continue
    with database.snapshot() as snapshot:
        for row in snapshot.execute_sql(f"SELECT id FROM {table_name}"):
            node_type_lookup[row[0]] = node_type

# ── Step 4: PyVis visualization ───────────────────────────────────
net = Network(
    notebook=True, directed=True,
    bgcolor="#222222", font_color="white",
    height="500px", width="100%", cdn_resources='remote'
)

for r in results:
    src_color = color_map.get(node_type_lookup.get(r["source"], ""), "#999999")
    tgt_color = color_map.get(node_type_lookup.get(r["target"], ""), "#999999")
    net.add_node(r["source"], label=r["source"], color=src_color)
    net.add_node(r["target"], label=r["target"], color=tgt_color)
    net.add_edge(r["source"], r["target"], title=r["label"], label=r["label"])

net.show("spanner_graph.html")
display(HTML("spanner_graph.html"))

print("\n─── To visualize in Spanner Studio, run this query ───")
print(f"""
GRAPH {SPANNER_GRAPH_NAME}
MATCH p = (a)-[e]->(b)
RETURN TO_JSON(p) AS path_json
LIMIT 50
""")

---
# 🔍 Section 3: The GraphRAG Advantage — Multi-Hop Reasoning

## The "Multi-Hop" Problem

Standard vector RAG treats your data like a pile of loose sticky notes. It searches for chunks of text mathematically similar to your query. But what happens when the answer is a puzzle with pieces hidden in different notes?

Consider:
> *"Which organization based in London is advising the architect at Acme Corp?"*

- Memo 2 tells us Sarah Jenkins is the architect at Acme Corp
- Memo 3 tells us Mark Chen advises Sarah Jenkins and works at Global Dynamics
- Memo 1 tells us Global Dynamics is in London

A vector search for *"organization in London advising Acme Corp architect"* will find Memos 2 and 3 but **Memo 1 has no keyword overlap** — so it gets dropped. The answer lives only if you connect all three.

## Step 3.1: Build the Semantic Entry Point

### The right way to do vector search in Spanner Graph

Spanner Graph stores embeddings **directly as a property on each node** — not in a separate table. This is the native approach.

`SpannerGraphVectorContextRetriever` handles the full hybrid search in one call:
```
User question → Embed → Vector search on graph nodes → Walk expand_by_hops outward → Return subgraph
```

### What `expand_by_hops` does

Setting `expand_by_hops=2` means:
- Vector search finds `Sarah Jenkins` (closest node to "the architect")
- The retriever walks **2 hops outward**: picks up `Acme Corp`, `Project Phoenix`, `Mark Chen`, `Global Dynamics`, `London`

The answer to almost any question about Sarah Jenkins is now in the returned subgraph — without writing a single GQL query.

### Step 3.1: Generate Rich Node Descriptions

**Why not just embed the node name?**

If we embed only `"Sarah Jenkins"`, the vector search will fail for queries like *"the architect"* because the word *architect* is not in the name.

Instead, we ask Gemini to write a **rich natural language description** of each node based on its **level-1 relationships only** — its direct connections.

**Why only level-1?**
If we include level-2+ relationships, Gemini might hallucinate connections. The graph traversal at query time handles multi-hop reasoning — the embedding only needs to make the node **findable**.

In [ ]:
# ── Fetch level-1 connections for each node ──────────────────────
# We read directly from the relational edge tables SpannerGraphStore created

print("⏳ Fetching level-1 connections for each node...\n")

# Build node → type lookup from tables that actually exist
node_type_lookup = {}
for node_type in allowed_nodes:
    table_name = f"{SPANNER_GRAPH_NAME}_{node_type}"
    check_sql  = f"SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '' AND TABLE_NAME = '{table_name}'"
    with database.snapshot() as snapshot:
        exists = list(snapshot.execute_sql(check_sql))[0][0] > 0
    if not exists:
        continue
    with database.snapshot() as snapshot:
        for row in snapshot.execute_sql(f"SELECT id FROM {table_name}"):
            node_type_lookup[row[0]] = node_type

print(f"✅ Nodes found: {node_type_lookup}\n")

# Build connections per node from the edge tables
node_connections = {node_id: {"outgoing": set(), "incoming": set()}
                    for node_id in node_type_lookup}

for table_name in edge_table_names:
    stripped  = table_name.replace(f"{SPANNER_GRAPH_NAME}_", "", 1)
    parts     = stripped.split("_")
    rel_label = "_".join(parts[1:-1]) if len(parts) > 2 else stripped

    with database.snapshot() as snapshot:
        for row in snapshot.execute_sql(f"SELECT id, target_id FROM {table_name}"):
            src, tgt = row[0], row[1]
            if src in node_connections:
                node_connections[src]["outgoing"].add(f"{rel_label} → {tgt}")
            if tgt in node_connections:
                node_connections[tgt]["incoming"].add(f"{src} → {rel_label}")

print("─── Level-1 connections per node ───")
for node_id, conns in node_connections.items():
    print(f"\n  [{node_id}]")
    print(f"    Outgoing: {list(conns['outgoing'])}")
    print(f"    Incoming: {list(conns['incoming'])}")

In [ ]:
# ── Ask Gemini to write a description for each node ──────────────
# Using ONLY level-1 connections — no multi-hop hallucination

print("⏳ Generating node descriptions with Gemini...\n")

node_descriptions = {}

for node_id, conns in node_connections.items():
    outgoing    = list(conns["outgoing"])
    incoming    = list(conns["incoming"])
    entity_type = node_type_lookup.get(node_id, "Entity")

    description_prompt = f"""You are building a search index for a knowledge graph.

    Here is a node and ONLY its direct level-1 connections:

    Node: {node_id}
    Type: {entity_type}
    Outgoing relationships: {outgoing if outgoing else 'none'}
    Incoming relationships: {incoming if incoming else 'none'}

    Write a 2-3 sentence natural language description of this entity.

    STRICT RULES:
    - Use ONLY the facts visible in the direct relationships listed above
    - Do NOT infer or assume connections that are not explicitly listed
    - Do NOT mention entities that do not appear in the lists above
    - Do NOT reason across multiple hops

    Write as natural flowing text, not bullet points."""

    description = extractor_llm.invoke(description_prompt).content.strip()
    node_descriptions[node_id] = description
    print(f"📝 {node_id}:\n   {description}\n")

print(f"✅ Generated descriptions for {len(node_descriptions)} nodes")

### Step 3.2: Write Embeddings onto Graph Nodes

Spanner Graph stores vector embeddings as a column directly on the node table — the same table that backs the graph node. This means embeddings and graph data are always in sync with no separate table to maintain.

The three steps are:
1. `ALTER TABLE` — add an `ARRAY<FLOAT64>` embeddings column to each node table
2. `UPDATE` — write the embedding value onto each node row
3. `CREATE OR REPLACE PROPERTY GRAPH` — refresh the graph schema so Spanner exposes the new column as a queryable node property
4. Initialise `SpannerGraphVectorContextRetriever` — it reads from that column at query time

In [ ]:
import vertexai
from vertexai.language_models import TextEmbeddingInput, TextEmbeddingModel

vertexai.init(project=GCP_PROJECT_ID, location=REGION)
embedding_model = TextEmbeddingModel.from_pretrained(EMBEDDING_MODEL)

def get_embedding(text):
    result = embedding_model.get_embeddings(
        [TextEmbeddingInput(task_type="RETRIEVAL_DOCUMENT", text=text)]
    )
    return result[0].values

# First add the embeddings column to each node table that exists
for node_type in allowed_nodes:
    table_name = f"{SPANNER_GRAPH_NAME}_{node_type}"
    # check if table exists first
    check_sql = f"SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '' AND TABLE_NAME = '{table_name}'"
    with database.snapshot() as snapshot:
        exists = list(snapshot.execute_sql(check_sql))[0][0] > 0
    if not exists:
        continue

    # Add the embeddings column if it doesn't already exist
    try:
        database.update_ddl([f"ALTER TABLE {table_name} ADD COLUMN IF NOT EXISTS embeddings ARRAY<FLOAT64>"])
        print(f"✅ Added embeddings column to {table_name}")
    except Exception as e:
        print(f"ℹ️  {table_name}: {e}")

# Now write descriptions + embeddings directly onto each node row
def write_embeddings(transaction):
    for node_id, description in node_descriptions.items():
        node_type = node_type_lookup.get(node_id)
        if not node_type:
            continue
        table_name = f"{SPANNER_GRAPH_NAME}_{node_type}"
        embedding  = get_embedding(description)

        transaction.execute_update(
            f"UPDATE {table_name} SET embeddings = @emb WHERE id = @node_id",
            params={"emb": embedding, "node_id": node_id},
            param_types={
                "emb":     spanner.param_types.Array(spanner.param_types.FLOAT64),
                "node_id": spanner.param_types.STRING,
            }
        )
        print(f"  ✅ Wrote embedding for [{node_id}]")

database.run_in_transaction(write_embeddings)
print("\n✅ Embeddings written directly to graph node tables")

### Step 3.3: Refresh the Property Graph Definition

When `SpannerGraphStore` created the Property Graph during `add_graph_documents`, it only knew about the columns that existed at that time — just `id`. Now that we have added an `embeddings` column to each node table, we need to tell Spanner Graph about it.

`CREATE OR REPLACE PROPERTY GRAPH` redefines the graph schema from scratch, picking up all current columns from the underlying tables. Two things are critical in this DDL:

1. **Explicit `LABEL` clause** — without it, Spanner uses the full table name as the node label (e.g. `corpgraph_Person` instead of `Person`), which breaks all GQL queries
2. **`PROPERTIES (id, embeddings)`** — explicitly exposes both columns as node properties so the retriever can search on `embeddings` and GQL queries can filter on `id`

This step must be re-run any time the underlying node tables change — for example if you add new properties or columns in a production pipeline.

In [ ]:
# After adding the embeddings column to the node tables,
# we must recreate the PROPERTY GRAPH definition so Spanner Graph
# exposes the new column as a node property.
# SpannerGraphStore created the original definition without this column,
# so the graph schema needs to be updated to reflect it.

print("⏳ Refreshing PROPERTY GRAPH definition...")

existing_node_tables = []
for node_type in allowed_nodes:
    table_name = f"{SPANNER_GRAPH_NAME}_{node_type}"
    check_sql  = f"SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '' AND TABLE_NAME = '{table_name}'"
    with database.snapshot() as snapshot:
        exists = list(snapshot.execute_sql(check_sql))[0][0] > 0
    if exists:
        existing_node_tables.append(table_name)

node_tables_parts = []
for table_name in existing_node_tables:
    label = table_name.replace(f"{SPANNER_GRAPH_NAME}_", "", 1)
    node_tables_parts.append(
        f"{table_name} LABEL {label} PROPERTIES (id, embeddings)"
    )
node_tables_clause = ",\n  ".join(node_tables_parts)

edge_tables_parts = []
for table_name in edge_table_names:
    stripped  = table_name.replace(f"{SPANNER_GRAPH_NAME}_", "", 1)
    parts     = stripped.split("_")
    src_type  = parts[0]
    tgt_type  = parts[-1]
    src_table = f"{SPANNER_GRAPH_NAME}_{src_type}"
    tgt_table = f"{SPANNER_GRAPH_NAME}_{tgt_type}"
    rel_label = "_".join(parts[1:-1])
    edge_tables_parts.append(
        f"{table_name}\n"
        f"    SOURCE KEY (id) REFERENCES {src_table} (id)\n"
        f"    DESTINATION KEY (target_id) REFERENCES {tgt_table} (id)\n"
        f"    LABEL {rel_label}"
    )
edge_tables_clause = ",\n  ".join(edge_tables_parts)

recreate_ddl = f"""
CREATE OR REPLACE PROPERTY GRAPH {SPANNER_GRAPH_NAME}
NODE TABLES (
  {node_tables_clause}
)
EDGE TABLES (
  {edge_tables_clause}
)
"""

print("DDL to be applied:")
print(recreate_ddl)

operation = database.update_ddl([recreate_ddl])
operation.result(120)
print("\n✅ PROPERTY GRAPH refreshed — node labels are now Person, Organization etc.")

### Step 3.4: Initialise the Vector Context Retriever

`SpannerGraphVectorContextRetriever` is a read-only retriever — it does not write any data. It must be initialised **after** the embeddings are written and the Property Graph is refreshed.

At query time it does two things in a single call:
1. **Vector search** — embeds the user question using `text-embedding-004` and finds the top-k graph nodes whose `embeddings` column is closest by cosine distance. This is how fuzzy natural language like *"the architect"* resolves to the exact node `Sarah Jenkins`.
2. **Graph expansion** — starting from those matched nodes, walks `expand_by_hops` hops outward along the graph edges, collecting all connected nodes and relationships into a subgraph.

The returned subgraph is what gets passed to Stage 2 — it gives Gemini the relevant slice of the graph to reason over when generating the GQL query.

In [ ]:
from langchain_google_spanner import SpannerGraphVectorContextRetriever
from langchain_google_vertexai import VertexAIEmbeddings

embedding_service = VertexAIEmbeddings(
    model_name=EMBEDDING_MODEL,
    project=GCP_PROJECT_ID,
    location=REGION
)

retriever = SpannerGraphVectorContextRetriever.from_params(
    graph_store=graph_store,
    embedding_service=embedding_service,
    embeddings_column="embeddings",   # matches the column we just added
    top_k=3,
    expand_by_hops=2,
)

print("✅ Retriever ready — embeddings already on graph nodes")

---
# 🏆 Section 4: The Query Pipeline

## How the pipeline works

```
User Question (natural language)
        │
        ▼
┌──────────────────────────────────────────┐
│  STAGE 1: Vector Search + Graph Expansion │
│  Embed the question                       │
│  → Find top-k closest nodes by embedding  │
│  → Walk expand_by_hops outward            │
│  → Returns: a relevant subgraph           │
└────────────────┬─────────────────────────┘
                 │ subgraph context
                 ▼
┌──────────────────────────────────────────┐
│  STAGE 2: GQL Generation                 │
│  Gemini receives question + subgraph     │
│  → Generates a precise GQL query         │
└────────────────┬─────────────────────────┘
                 │ GQL query
                 ▼
┌──────────────────────────────────────────┐
│  STAGE 3: Execution & Synthesis          │
│  Execute GQL against Spanner Graph       │
│  Gemini synthesizes a natural language   │
│  answer from the result                  │
└──────────────────────────────────────────┘
```

### The Key Insight: Two Different Roles
- **Vector search + graph expansion** answers: *"Which nodes and their neighbourhood are relevant?"*
- **GQL generation** answers: *"What is the precise logical path through those nodes?"*
- **Synthesis** answers: *"What does this mean in plain English?"*

## Step 4.1: Fetch the Graph Schema

Before running queries we need to give Gemini an accurate schema so it generates valid GQL.

**Important:** We read the schema from `INFORMATION_SCHEMA` — not from GQL — because `SpannerGraphStore` encodes the relationship type in the **edge table name**, not as a column on the edge. This is why we can't use `edge.label` in GQL queries.

In [ ]:
# Build graph schema dynamically from INFORMATION_SCHEMA
# No hardcoded node types, labels, or relationship types

# Node labels — read from tables that actually exist
node_labels = []
for node_type in allowed_nodes:
    table_name = f"{SPANNER_GRAPH_NAME}_{node_type}"
    check_sql  = f"SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '' AND TABLE_NAME = '{table_name}'"
    with database.snapshot() as snapshot:
        exists = list(snapshot.execute_sql(check_sql))[0][0] > 0
    if exists:
        node_labels.append(node_type)

# Relationship types and their source/target — derived from edge table names
# Table name pattern: {graph}_{SourceType}_{REL}_{TargetType}
relationships = []
for table_name in edge_table_names:
    stripped  = table_name.replace(f"{SPANNER_GRAPH_NAME}_", "", 1)
    parts     = stripped.split("_")
    src_type  = parts[0]
    tgt_type  = parts[-1]
    rel_label = "_".join(parts[1:-1])
    relationships.append({
        "source":    src_type,
        "label":     rel_label,
        "target":    tgt_type,
    })

# Build the schema string
rel_lines = "\n".join([
    f"- ({r['source']})-[{r['label']}]->({r['target']})"
    for r in relationships
])

graph_schema = f"""
Graph: {SPANNER_GRAPH_NAME}

Node labels: {node_labels}
Node property: id (STRING) — unique identifier

Relationships:
{rel_lines}

Note: treat PARTNERED_WITH as bidirectional — use -[e:PARTNERED_WITH]- (no arrow)
"""

print("📋 Graph Schema:")
print(graph_schema)

In [ ]:
# ── Pipeline functions ───────────────────────────────────────────

def stage1_retrieve_subgraph(user_question: str) -> list:
    """
    Stage 1: SpannerGraphVectorContextRetriever embeds the question,
    finds the closest graph nodes, then automatically walks expand_by_hops
    outward to return the relevant subgraph as LangChain Documents.
    """
    return retriever.invoke(user_question)


def stage2_generate_gql(user_question: str, subgraph_docs: list) -> str:
    subgraph_context = "\n".join([doc.page_content for doc in subgraph_docs])

    gql_prompt = f"""You are an expert in GQL (Graph Query Language) for Google Cloud Spanner Graph.

    User Question: {user_question}

    Relevant subgraph context retrieved via vector search:
    {subgraph_context}

    Graph Schema:
    {graph_schema}

    GQL SYNTAX RULES:

    1. Start every query with: GRAPH {SPANNER_GRAPH_NAME}

    2. Use a single MATCH statement with all patterns separated by commas.

    3. Edge pattern syntax: (a:LabelA)-[e:RELATIONSHIP]->(b:LabelB)
      The edge variable is mandatory. Use labels and relationships from the schema only.

    4. For bidirectional relationships, omit the arrow: (a)-[e:RELATIONSHIP]-(b)

    5. To traverse a relationship in reverse, flip the arrow direction:
      (b:LabelB)<-[e:RELATIONSHIP]-(a:LabelA)

    6. One WHERE clause only, placed after the entire MATCH pattern.
      Combine all filters using AND.

    7. Use property dot notation for filters: variable.id = 'value'

    8. End with a RETURN clause using descriptive column aliases.

    Output ONLY the GQL query. No explanation. No markdown fences."""

    raw = extractor_llm.invoke(gql_prompt).content.strip()
    return raw.replace("```gql", "").replace("```sql", "").replace("```", "").strip()

def stage3_execute_and_synthesize(user_question: str, gql_query: str) -> tuple:
    results = []
    try:
        with database.snapshot() as snapshot:
            rows = snapshot.execute_sql(gql_query)
            fields = None
            for row in rows:
                # fields only becomes available after first row is fetched
                if fields is None:
                    fields = [field.name for field in rows.fields]
                results.append(dict(zip(fields, row)))
    except Exception as e:
        return [], f"❌ GQL Execution Error: {e}"

    if not results:
        return [], "⚠️  No results found."

    synthesis_prompt = f"""The user asked: "{user_question}"

A graph database query was executed and returned this result: {results}

This result IS the direct answer to the question.
State the answer clearly and confidently in one sentence.
Do not say the data is missing or incomplete."""

    answer = extractor_llm.invoke(synthesis_prompt).content.strip()
    return results, answer


print("✅ Pipeline functions ready!")

---
# 🧪 Section 5: Query Tests

We now run the full pipeline end-to-end. Watch the output carefully:
- **Stage 1** shows which subgraph the retriever returned
- **Stage 2** shows the GQL query Gemini generated — notice how it walks the path
- **Stage 3** shows the raw Spanner result and the final synthesized answer

## Query 1: Simple Multi-Hop

> *"Which organization based in London is advising the architect at Acme Corp?"*

This requires 4 hops:
```
Acme Corp → WORKS_AT ← Sarah Jenkins → ADVISES ← Mark Chen → WORKS_AT → Global Dynamics → LOCATED_IN → London
```

In [ ]:
user_query = "Which organization based in London is advising the architect at Acme Corp?"

print(f"🤔 User Query: {user_query}\n")
print("=" * 65)

# Stage 1
print("\n📡 STAGE 1: Vector Search + Graph Expansion")
subgraph_docs = stage1_retrieve_subgraph(user_query)
print(f"   Retrieved {len(subgraph_docs)} documents from subgraph")
for doc in subgraph_docs:
    print(f"   → {doc.page_content[:100]}...")

# Stage 2
print("\n⚙️  STAGE 2: GQL Generation (Gemini)")
gql_query = stage2_generate_gql(user_query, subgraph_docs)
print("   Generated GQL:")
for line in gql_query.split("\n"):
    print(f"   {line}")

# Stage 3
print("\n🔄 STAGE 3: Execution & Synthesis")
results, final_answer = stage3_execute_and_synthesize(user_query, gql_query)
print(f"   Raw graph result: {results}")

print("\n" + "=" * 65)
print(f"\n💡 FINAL ANSWER:\n   {final_answer}")

## Query 2: The 6-Hop Stress Test

> *"Which project is being led by the person that the London-based partner of the San Francisco company is advising?"*

**The path the system must walk:**
```
1. Find org located in San Francisco  → Acme Corp
2. Find Acme Corp's partner           → Global Dynamics
3. Verify Global Dynamics is London   → ✅
4. Find who works at Global Dynamics  → Mark Chen
5. Find who Mark Chen advises         → Sarah Jenkins
6. Find Sarah Jenkins's project       → Project Phoenix ✅
```

**Note:** The answer `Project Phoenix` never appears in the same sentence as `San Francisco` or `London` in the original memos. The answer is **assembled** by the graph — it doesn't exist anywhere as a pre-written fact.

In [ ]:
hard_query = "Which project is being led by the person that the London-based partner of the San Francisco company is advising?"

print(f"🤔 User Query: {hard_query}\n")
print("=" * 70)

# Stage 1
print("\n📡 STAGE 1: Vector Search + Graph Expansion")
subgraph_docs = stage1_retrieve_subgraph(hard_query)
print(f"   Retrieved {len(subgraph_docs)} documents from subgraph")
for doc in subgraph_docs:
    print(f"   → {doc.page_content[:100]}...")

# Stage 2
print("\n⚙️  STAGE 2: GQL Generation (Gemini)")
gql_query = stage2_generate_gql(hard_query, subgraph_docs)
print("   Generated GQL:")
for line in gql_query.split("\n"):
    print(f"   {line}")

# Stage 3
print("\n🔄 STAGE 3: Execution & Synthesis")
results, final_answer = stage3_execute_and_synthesize(hard_query, gql_query)
print(f"   Raw graph result: {results}")

print("\n" + "=" * 70)
print(f"\n💡 FINAL ANSWER:\n   {final_answer}")

---
# 🛠️ Section 6: Using SpannerGraphQAChain (Simpler API)

The pipeline we built manually above gives you **full visibility and control** over each stage. But LangChain also provides `SpannerGraphQAChain` — a higher-level abstraction that handles the LLM → GQL → answer loop automatically.

| Use | When |
|---|---|
| Manual pipeline (Sections 4-5) | Fuzzy natural language, need to debug GQL, full control |
| `SpannerGraphQAChain` | Simple Q&A with exact entity names, minimal code |

In [ ]:
from langchain_google_spanner import SpannerGraphQAChain
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import PromptTemplate

llm_for_chain = ChatVertexAI(model=MODEL_NAME, temperature=0)

chain = SpannerGraphQAChain.from_llm(
    llm_for_chain,
    graph=graph_store,
    allow_dangerous_requests=True,
    verbose=True,
    return_intermediate_steps=True,
    # gql_prompt=gql_prompt,
)

exact_query = "What does Mark Chen do at Global Dynamics?"
print(f"🤔 Query: {exact_query}")
response = chain.invoke("query=" + exact_query)
print(f"\n💡 Answer: {response['result']}")

---
# 📖 Section 7: Summary

## What We Built

| Step | What we did | Technology |
|------|-------------|------------|
| 1 | Provisioned graph database | Cloud Spanner + Spanner Graph |
| 2 | Extracted entities + relationships | Gemini + LLMGraphTransformer |
| 3 | Stored the Knowledge Graph | SpannerGraphStore (LangChain) |
| 4 | Generated rich node descriptions | Gemini (level-1 context only) |
| 5 | Indexed embeddings on graph nodes | SpannerGraphVectorContextRetriever |
| 6 | Built hybrid search pipeline | Vector expansion + GQL generation + synthesis |
| 7 | Ran 6-hop multi-hop queries | Spanner Graph + GQL |

## Key Design Decisions

**Why level-1-only node descriptions?**
Prevents hallucination of multi-hop facts in embeddings. The embedding only needs to make the node findable — multi-hop reasoning belongs to the graph traversal.

**Why hybrid (vector + graph)?**
Vector search alone can't follow relationship chains. Graph traversal alone can't resolve fuzzy natural language to exact node IDs. Together they handle the full spectrum.

---

## Next Steps
- 📚 [LangChain + Spanner docs](https://github.com/googleapis/langchain-google-spanner-python)
- 🔍 [Cloud Spanner Graph docs](https://cloud.google.com/spanner/docs/graph)
- 🤖 [Vertex AI Agent Engine](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview)
- 📊 [Spanner Graph Notebook explorer](https://github.com/cloudspannerecosystem/spanner-graph-notebook)
- 🛡️ [Spanner IAM and access control](https://cloud.google.com/spanner/docs/iam)

---
# 🏆 Hackathon Challenge

## The New Dataset

Three new memos have been added to the corporate network. Your job is to:
1. Ingest them into your Spanner Graph existing dataset( DO NOT remove old data )
2. Use your pipeline to answer two multi-hop questions and run code below

Submit the output of all three tasks in the survey link at the end.

---

## Additional Memos

```
Memo 4: Priya Patel is the Chief Risk Officer at Nexus Bank.
Nexus Bank is a financial institution headquartered in Singapore.

Memo 5: Nexus Bank has formed a strategic partnership with Acme Corp
to co-develop an AI-powered risk assessment platform called Project Shield.
Priya Patel is the executive sponsor of Project Shield.

Memo 6: James Liu is a senior consultant at Global Dynamics.
He has been assigned to advise Priya Patel on regulatory compliance matters.
Global Dynamics has also opened a new office in Singapore.
```

## ✅ Task 1: Ingestion verification code

Add the new memos to your Spanner Graph using the same pipeline you built.
Then run the verification code below to prove the data landed correctly.

In [ ]:
with database.snapshot() as snapshot:
    rows   = snapshot.execute_sql(f"""
        GRAPH {SPANNER_GRAPH_NAME}
        MATCH (a)-[e]->(b)
        RETURN a.id AS source, b.id AS target
        ORDER BY source
    """)
    fields = None
    count  = 0
    for row in rows:
        if fields is None:
            fields = [f.name for f in rows.fields]
        r = dict(zip(fields, row))
        print(f"({r['source']}) → ({r['target']})")
        count += 1

## 🔍 Task 2

> **"Which consultant from Global Dynamics is advising the executive sponsor of the project co-developed with Acme Corp?"**

Run your pipeline with this question and paste the final answer in the survey.

📋 **Submit the FINAL ANSWER in the survey.**

## 🔍 Task 3

> **"Which city is home to both a Global Dynamics office and the headquarters of the bank that partnered with Acme Corp?"**

Run your pipeline with this question and paste the final answer in the survey.

📋 **Submit the FINAL ANSWER in the survey.**

---

## 📝 Submit Your Results

**[👉 Submit Survey](https://forms.gle/68Brzh5kUZ6mqcsh7)**

Include in your submission:
- **Task 1:** The full verification output
- **Task 2:** The final answer from your pipeline
- **Task 3:** The final answer from your pipeline